# `ClassBase`: learning an unfamiliar Nematics3D object

`nematics3d.core.class_base.ClassBase` is a low-level base class shared by many objects in Nematics3D. Users normally do **not** create a `ClassBase` directly. Instead, concrete classes inherit from it and therefore share a common way to describe themselves, organize attributes, expose actions, and record relations to other objects.

The practical consequence is simple: once you learn the small common vocabulary supplied by `ClassBase`, you do not need to memorize the full API of every Nematics3D object before you can start using it.

This tutorial starts from the most realistic situation: **you have just received an object and have almost no idea what it is or how to use it.**


## Setup: an object we have never used before

We will use `SmoothedLine` as the example. Its scientific role is simple enough here: it stores a polyline and constructs a smoothed version of it.


In [1]:
import numpy as np
import nematics3d as n3d

t = np.linspace(0.0, 4.0 * np.pi, 101)
coords = np.column_stack([
    t,
    np.sin(t) + 0.08 * np.sin(9.0 * t),
    0.25 * np.cos(0.5 * t),
])

line = n3d.SmoothedLine(
    coords,
    window_length=11,
)

line


SmoothedLine('line')

## I have this object. What do I do with it?

Suppose `line` was handed to you by somebody else. You may not know what class it belongs to, what data it contains, which fields are important, or what you are allowed to change.

A good first move in an editor, IPython, or Jupyter notebook is simply to type:

```python
line.show_
```

and invoke autocomplete, for example by pressing **Tab**.

The `show_` prefix groups the object's inspection and explanation methods. In other words, when you do not yet know an object, **ask the object to explain itself**.

Some of the most useful entries are:

| Method | Question it answers |
| --- | --- |
| `show_doc()` | What are you? What are you for? |
| `show_readable_attrs()` | What information do you contain? |
| `show_attr_doc(...)` | What does this particular attribute mean? |
| `show_attr_info(...)` | Give me detailed information about this attribute. |
| `show_relations()` | What other objects are you related to? |
| `show_relation_tree()` | How are those object relations organized? |


## First question: what are you?

Start with the most basic question:


In [2]:
line.show_doc()


[INFO]
    <show_doc> 
    SmoothedLine wraps a polyline and optionally produces a smoothed result.
    
    This class keeps raw input coordinates together with a smoothing pipeline,
    a cached spline representation, and the final resampled output line.
    Normal users provide input coordinates, then inspect or change smoothing
    settings through `line.opts` or `line.act_commit(...)`.
    
    Important readable attributes:
    
    - `opts`: the paired OptsSmoothedLine controlling the smoothing pipeline.
    - `raw_coords`: the original input polyline coordinates.
    - `calc_coords`: the processed coordinates currently entering the smoother.
    - `calc_num_init`: the number of processed input points currently used.
    - `calc_num_out`: the number of output points requested after smoothing.
    - `result`: the final output coordinates, either smoothed or fallback raw
      coordinates.
    - `entity_tck`: the spline cache used for tangent evaluation, or None when
      smoothi

`show_doc()` displays the class docstring of the **concrete class of this object**. Here it describes `SmoothedLine`, not `ClassBase`.

This is the quickest way to learn the object's overall role before looking at individual fields.


## Second question: what do you contain?

Once you know the object's general purpose, ask what user-readable information it exposes:


In [3]:
line.show_readable_attrs()


[INFO]
    <show_readable_attrs> 
    When reading host fields, the 'raw_' prefix may be omitted where a public alias exists.
    'attrs_forbidden': Read-only union of wrapped attrs and host-declared protected attrs.
    'attrs_protected': Read-only: Public attrs currently marked as directly protected on this host.
    'attrs_wrapped': Read-only: Public attrs currently blocked because they are wrapped.
    'calc_coords': The processed coordinates actually sent into the smoothing pipeline
    'calc_is_smoothed': Boolean flag indicating whether smoothing was applied
    'calc_num_init': Read-only: Number of processed input points currently entering the smoothing pipeline.
    'calc_num_out': Read-only: Number of output points requested after smoothing.
    'calc_result': The smoothed output coordinates (shape: M x D)
    'calc_status': Status indicator of the smoothing pipeline. Set to 'success' if smoothing completes normally. If smoothing is skipped or disabled due to internally detect

The result is more useful than a raw dump of Python internals: it presents the attributes that belong to the object's public vocabulary together with their descriptions.

If one name catches your attention, inspect only that field. For example:


In [4]:
line.show_attr_doc("raw_coords")


[INFO]
    <show_attr_doc> 
    Raw input line coordinates (shape: N x D)


If you want more than the documentation string, use:


In [5]:
line.show_attr_info("raw_coords")


[INFO]
    <show_attr_info> 
    name: raw_coords
    kind: raw
    alias: coords
    modifiable: yes
    protected: no
    value: array([[ 0.00000000e+00,  0.00000000e+00,  2.50000000e-01],        [ 1.25663706e-01,  1.97719398e-01,  2.49506682e-01],        [ 2.51327412e-01,  3.10330947e-01,  2.48028675e-01],        [ 3.76991118e-01,  3.48229362e-01,  2.45571813e-01...
    doc: Raw input line coordinates (shape: N x D)


`show_attr_info(...)` combines information such as the canonical name, attribute kind, whether the field is modifiable or protected, its current value, and its documentation.

At this point, without reading the implementation, we already have a workable path through an unfamiliar object:

```text
line.show_  + autocomplete
  -> line.show_doc()
  -> line.show_readable_attrs()
  -> line.show_attr_doc("...")
  -> line.show_attr_info("...")
```


## Why does this discovery workflow work so well? Prefixes are part of the API

The fact that typing `show_` immediately finds the explanation tools is not accidental. `ClassBase` uses explicit naming conventions so that names carry semantic information.

There are two closely related conventions:

- **method prefixes** tell you what kind of operation a method performs;
- **attribute prefixes** tell you what role a piece of data plays in the object.

This means autocomplete is not merely a convenience. It is one of the intended ways to discover a Nematics3D object's interface.


### Method prefixes: `show_` and `act_`

Public object methods commonly use two semantic prefixes:

| Prefix | Meaning | Typical question |
| --- | --- | --- |
| `show_...` | Inspect, display, or explain something | What can you tell me? |
| `act_...` | Perform an action on or through the object | What can you do? |

You have already used the first one:

```python
line.show_      # autocomplete: what can this object show or explain?
```

The same idea works for actions:

```python
line.act_       # autocomplete: what actions can this object perform?
```

The important point is that `act_` names tell you that these methods **do something**, rather than merely report information.


### Attribute prefixes: what role does this value play?

Most structured `ClassBase` attributes belong to a semantic category indicated by their name:

| Prefix | Meaning | A useful way to read it |
| --- | --- | --- |
| `raw_...` | Canonical stored public input or base data | What was given to the object? |
| `state_...` | Writable runtime state | What state is the object currently in? |
| `default_...` | Managed default-layer input | What default value/settings does it carry? |
| `calc_...` | Computed readable data, normally read-only | What has the object calculated? |
| `entity_...` | Computed or generated object-valued result, normally read-only | What object has it created? |

There are also deliberately unprefixed public forms, notably semantic relations such as `owner` or `registry`, ordinary Python properties defined by a concrete class, and user-added extra attributes. Their meaning is documented separately rather than inferred from one of the prefixes above.


## Use the prefixes to explore instead of memorizing names

Suppose you know that `line` has calculated several quantities, but you do not remember their names. Type:

```python
line.calc_
```

and invoke autocomplete. For `SmoothedLine`, this narrows the candidates to calculated fields such as `calc_coords`, `calc_result`, and `calc_status`.

The same pattern answers several common questions:

```python
line.raw_       # What base input does this object store?
line.state_     # What runtime state does it expose?
line.default_   # What default-layer values does it expose?
line.calc_      # What has it calculated?
line.entity_    # What object-valued results has it generated?
line.show_      # How can it explain itself?
line.act_       # What can it do?
```

A useful rule of thumb is therefore:

> **Do not start by memorizing every concrete class. Learn the `ClassBase` vocabulary, then let autocomplete show you which words this particular object provides.**


## Relations: the important case outside the prefix system

At this point, `show_...` methods together with attribute prefixes already cover most of what you need to discover and understand an unfamiliar object's attributes. In most cases, you can inspect what the object contains, identify the role of a field from its prefix, and ask for detailed information without knowing the class in advance.

There is, however, one important category that sits somewhat outside this prefix-based attribute system: **relations**. Relations describe how this object is semantically connected to other objects, for example through an `owner` or a `registry`. Because these connections do not fit naturally into `raw_`, `state_`, `default_`, `calc_`, or `entity_`, it is worth learning them separately.

A `SmoothedLine` has only a small relation structure, so for this section we switch to a more realistic object graph produced by `quick_visualize_q()`. The bundled Q-tensor example is the same data used by the dedicated quick-visualization tutorial. This workflow automatically creates an `n-plane` object that is connected to its grid, interpolator, Q field, figure, visual glyphs, bounds, and registries, making it a compact but genuinely rich example of relations.


In [6]:
from pathlib import Path

repo_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "example" / "data" / "Q_example_workflow.npy").exists()
)
Q_data = np.load(repo_root / "example" / "data" / "Q_example_workflow.npy")

Q_obj, figure = n3d.quick_visualize_q(Q=Q_data)
n_plane = Q_obj.objs["n-plane"]


[PROGRESS]
        <QFieldObject.__init__> 
        Start to initialize Q tensor `Q`.
[PROGRESS]
        <QFieldObject.__init__> 
        Start defect analysis as detecting defects and classifying them into distinct lines for Q tensor `Q` 
        This operation might take a while.
        You can disable this automatic operation by setting is_detect_defects=False and is_classify_lines=False when initializing the Q tensor.
[INFO]
            <QFieldObject[name='Q'].act_defect_detect> 
            1270 defects are found.
[INFO]
            <QFieldObject[name='Q'].act_lines_classify> 
            8 lines are found.
[PROGRESS]
        <QFieldObject.__init__> 
        Defect analysis is finished, with 0.07 s
[INFO]
        <QFieldObject[name='Q'].act_lines_smooth> 
        There are 8 disclination lines in total, with 7 lines are smoothed.
        The smoothing window length is: 41


First inspect only the relations directly attached to this plane object:


In [7]:
n_plane.show_relations()


[INFO]
    <show_relations> 
    - registry
          The Registry object where this instance is registered.
          current: RegistryBase('objects manager')
    - grid
          The plane grid associated with this interpolated field.
          current: PlaneGrid('n-plane-grid')
    - interpolator
          The grid interpolator object used to sample this plane.
          current: GridInterpolator('Q interpolator')
    - visual_nb
          The PlotRod visual showing directors in the bulk region of this Q plane.
          current: PlotRod("n bulk of plane 'n-plane'")
    - visual_nd
          The PlotRod visual showing directors near detected defects on this Q plane.
          current: PlotRod("n near defect of plane 'n-plane'")
    - visual_defect
          The PlotSphere visual showing detected defect positions on this Q plane.
          current: PlotSphere("defects of plane 'n-plane'")


`show_relations()` stops at this first layer. To follow those related objects and see how the larger object graph is connected, use `show_relation_tree()`. Here we explicitly use `depth=3` so that both the breadth and the nested structure are visible:

```python
n_plane.show_relation_tree(depth=3)
```

The result is displayed as a tree, but the underlying structure is a graph: different paths can lead back to an object that has already been visited. Such repeated nodes are marked with `[visited]` rather than expanded forever.

This is the key distinction: **prefixes and attribute-inspection methods tell you what one object contains; relations tell you how that object participates in a larger object system.** Relations describe semantic object connections, not an automatic recomputation contract. Any synchronization or dependency behavior is defined separately by the relevant concrete classes.


## A practical discovery workflow

When you encounter an unfamiliar Nematics3D object, you can usually begin with the following sequence:

```python
obj.show_                     # autocomplete: how can the object explain itself?
obj.show_doc()                # what is it?
obj.show_readable_attrs()     # what does it contain?
obj.show_attr_info("...")   # what exactly is this field?

obj.raw_                      # autocomplete: base inputs
obj.state_                    # autocomplete: runtime state
obj.default_                  # autocomplete: default-layer values
obj.calc_                     # autocomplete: calculated data
obj.entity_                   # autocomplete: generated objects
obj.act_                      # autocomplete: available actions
```

If the object participates in a larger object structure, also try:

```python
obj.show_relations()
obj.show_relation_tree()
```


## The main idea

`ClassBase` is not something most users need to instantiate. It is the common object protocol underneath many Nematics3D classes.

The most useful habit is therefore not to memorize every class-specific attribute. Instead:

1. **Start with `show_` and autocomplete.** Let the object tell you what it is and what it contains.
2. **Read prefixes as semantic information.** `raw_`, `state_`, `default_`, `calc_`, and `entity_` tell you what role an attribute plays.
3. **Use autocomplete as API discovery.** `calc_`, `entity_`, `show_`, and `act_` immediately narrow a large object to the category you care about.

Once this vocabulary becomes familiar, a new Nematics3D object is no longer a completely new API. You already know how to start asking it the right questions.
